# Small FastSA Fill Example (qft_5.qasm)

This notebook runs `Fastsa_Fill` from `naive_n_dag` directly to place qubits on a grid.
The placement cost is the full schedule-time model, so the config needs the device and
timing parameters; it uses `benchmarks/configs/qft/qft_06_n_fastsa.json` (a 3x3 trap
grid) and parses its unit strings the same way `naive_n_dag.main` does. Nothing is
written to disk.

In [1]:
from pathlib import Path

from naive_n_dag.dag_helper import load_qasm_to_two_qubit_dag_with_single_qubit_context
from naive_n_dag.grid import Fastsa_Fill, generate_grid
from naive_n_dag.main import _coerce_unit_config, load_config


In [2]:
repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "naive_n_dag").is_dir())
qasm_path = repo_root / "inputs" / "qasm_files" / "qft" / "qft_5.qasm"
two_qubit_dag, _single_layers = load_qasm_to_two_qubit_dag_with_single_qubit_context(qasm_path)

config = load_config(repo_root / "benchmarks" / "configs" / "qft" / "qft_06_n_fastsa.json")
_coerce_unit_config(config)  # unit strings -> quantities, as naive_n_dag.main does
num_qubits = len(two_qubit_dag.qubits)
grid = generate_grid(config["dimensions"], config["lattice_spacing"])  # 3x3 traps -> 5x5 grid

qubits, result = Fastsa_Fill(
    grid,
    num_qubits,
    two_qubit_dag,
    config,
    seed=0,
    stage1_iterations=8,
    stage2_iterations=20,
    stage3_max_iterations=100,
    stage3_temperature_threshold=1e-4,
    return_result=True,
)

layout = {q.id: q.grid_position() for q in qubits}
print(f"QASM: {qasm_path.relative_to(repo_root)}")
print(f"Qubits: {num_qubits}")
print(f"Best cost (estimated schedule time, s): {result.best_cost * result.cost_scale:.6g}")
print("Grid positions (row, col):")
layout

QASM: inputs/qasm_files/qft/qft_5.qasm
Qubits: 5
Best cost (estimated schedule time, s): 3926.24
Grid positions (row, col):


{0: (2, 2), 1: (4, 2), 2: (2, 4), 3: (4, 0), 4: (2, 0)}